In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# @title

import os
import cv2
import json
from tqdm import tqdm
import albumentations as A
import random

class RandomizedImageAugmenter:
    def __init__(self, num_aug=5):
        self.num_aug = num_aug

    def _get_aug_pipeline(self):
        br = 0.12
        ct = 0.12
        hue = 5
        sat = 8
        val = 8

        shift = 0.04
        scale = 0.04
        rotate = 12

        return A.Compose([
            # 1) Rotations an toàn
            A.Rotate(limit=(-12, 12), border_mode=cv2.BORDER_REFLECT_101, p=1.0),

            # 2) Photometric (nhẹ – không phá chi tiết)
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=br, contrast_limit=ct, p=1),
                A.HueSaturationValue(hue, sat, val, p=0.3),
                A.ColorJitter(
                    brightness=(1 - br, 1 + br),
                    contrast=(1 - ct, 1 + ct),
                    saturation=(0.7, 1.0),
                    hue=(-0.03, 0.03),
                    p=0.3
                ),
            ], p=0.7),

            # 3) Contrast/Gamma bổ sung nhẹ
            A.CLAHE(clip_limit=2, p=0.5),
            A.RandomGamma(gamma_limit=(95, 110), p=0.5),

            # 4) Blur + Noise nhẹ
            # A.OneOf([
            #     A.GaussianBlur(blur_limit=1),
            #     A.MedianBlur(blur_limit=1),
            # ], p=0.10),

            A.OneOf([
                A.GaussNoise(var_limit=(0.5, 2)),          # noise rất nhẹ
                A.ISONoise(color_shift=(0.005, 0.015)),    # gần như không đổi màu
            ], p=0.3),

            # 5) Geometric (siêu nhẹ)
            A.ShiftScaleRotate(
                shift_limit=shift,
                scale_limit=scale,
                rotate_limit=rotate,
                border_mode=cv2.BORDER_REFLECT_101,
                p=0.3
            ),

            # 6) Flip
            A.HorizontalFlip(p=0.5),
        ])




    def augment_dataset(self, img_dir, anno_path, out_img_dir, out_anno_path):
        os.makedirs(out_img_dir, exist_ok=True)

        # ---------------------------
        # 1) Load annotation gốc
        # ---------------------------
        with open(anno_path, "r") as f:
            coco = json.load(f)

        # ---------------------------
        # 2) Nếu có annotation OUT → append tiếp
        # ---------------------------
        if os.path.exists(out_anno_path):
            print("📌 Output annotation found → APPEND mode")
            with open(out_anno_path, "r") as f:
                new_coco = json.load(f)

            # Lấy ID tiếp theo
            img_id_new = max([img["id"] for img in new_coco["images"]], default=0) + 1
            ann_id_new = max([ann["id"] for ann in new_coco["annotations"]], default=0) + 1

        else:
            print("🆕 No output annotation found → CREATE NEW")
            new_coco = {"images": [], "annotations": [], "categories": coco["categories"]}
            img_id_new = 1
            ann_id_new = 1

        # Gom annotation theo image
        ann_by_img = {}
        for ann in coco["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)

        # ---------------------------
        # 3) LOOP QUA ẢNH GỐC
        # ---------------------------
        for img_info in tqdm(coco["images"], desc="Augmenting"):
            image_id = img_info["id"]
            file_name = img_info["file_name"]
            img_path = os.path.join(img_dir, file_name)
            img = cv2.imread(img_path)
            if img is None:
                print("⚠ Không đọc được ảnh:", img_path)
                continue

            # ---- Lưu ảnh gốc ----
            out_name = f"{file_name.split('.')[0]}_orig.jpg"
            cv2.imwrite(os.path.join(out_img_dir, out_name), img)
            new_coco["images"].append({
                "id": img_id_new,
                "file_name": out_name,
                "width": img_info["width"],
                "height": img_info["height"]
            })

            if image_id in ann_by_img:
                for ann in ann_by_img[image_id]:
                    new_ann = ann.copy()
                    new_ann["id"] = ann_id_new
                    new_ann["image_id"] = img_id_new
                    new_coco["annotations"].append(new_ann)
                    ann_id_new += 1

            img_id_new += 1

            # ---- Augment ----
            for i in range(self.num_aug):
                aug_pipeline = self._get_aug_pipeline()
                img_aug = aug_pipeline(image=img)["image"]

                aug_name = f"{file_name.split('.')[0]}_aug{i+1}.jpg"
                cv2.imwrite(os.path.join(out_img_dir, aug_name), img_aug)

                new_coco["images"].append({
                    "id": img_id_new,
                    "file_name": aug_name,
                    "width": img_info["width"],
                    "height": img_info["height"]
                })

                if image_id in ann_by_img:
                    for ann in ann_by_img[image_id]:
                        new_ann = ann.copy()
                        new_ann["id"] = ann_id_new
                        new_ann["image_id"] = img_id_new
                        new_coco["annotations"].append(new_ann)
                        ann_id_new += 1

                img_id_new += 1

        # ---------------------------
        # 4) Lưu annotation OUT
        # ---------------------------
        with open(out_anno_path, "w") as f:
            json.dump(new_coco, f, indent=2)

        print("\n🎉 DONE! Dataset augmented and APPENDED successfully.")


# ===========================
# MAIN EXECUTION
# ===========================
if __name__ == '__main__':
    augmenter = RandomizedImageAugmenter(num_aug=0)
    augmenter.augment_dataset(
        img_dir='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train/Images',
        anno_path='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train/_annotations.coco.json',
        out_img_dir='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/Images',
        out_anno_path='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/_annotations.coco.json'
    )


📌 Output annotation found → APPEND mode


Augmenting: 100%|██████████| 207/207 [01:54<00:00,  1.81it/s]


🎉 DONE! Dataset augmented and APPENDED successfully.


In [ ]:
import os
import cv2
import json
import matplotlib.pyplot as plt

def load_coco_annotations(json_path):
    with open(json_path, "r") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}

    annos = {}
    for ann in coco["annotations"]:
        annos.setdefault(ann["image_id"], []).append(ann)

    return images, annos


def draw_keypoints(img, keypoints, radius=3, color=(255, 0, 0)):
    keypoints = list(keypoints)
    pts = [keypoints[i:i+3] for i in range(0, len(keypoints), 3)]

    for x, y, v in pts:
        if v > 0:
            cv2.circle(img, (int(x), int(y)), radius, color, -1)

    return img


def visualize_multiple_vertical(img_dir, json_path, max_images=6):
    images, annos = load_coco_annotations(json_path)

    image_ids = list(images.keys())[:max_images]
    n = len(image_ids)

    rows = n
    cols = 1

    plt.figure(figsize=(6, 3 * n))

    for idx, img_id in enumerate(image_ids):
        info = images[img_id]
        path = os.path.join(img_dir, info["file_name"])

        img = cv2.imread(path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if img_id in annos:
            for ann in annos[img_id]:
                img = draw_keypoints(img, ann["keypoints"])

        # Resize ảnh để không quá to
        scale = 500 / max(img.shape[0], img.shape[1])
        img_small = cv2.resize(img, None, fx=scale, fy=scale)

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img_small)
        plt.title(info["file_name"], fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


visualize_multiple_vertical(
    img_dir="/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/Images",
    json_path="/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/_annotations.coco.json",
    max_images=350
)


Output hidden; open in https://colab.research.google.com to view.